# **MÓDULO 17 - Projeto de Credit Score - Parte 1 - Processamento dos dados**


Essa é a primeira etapa do processo de Credit Score que vocês desenvolverão durante nosso curso.
Nessa primeira etapa vocês irão aplicar os passos aprendidos nos módulos de pré processamento para preparar a base de vocês para o desenvolvimento do modelo.

O termo "credit score" se refere a uma pontuação numérica que representa a credibilidade de um indivíduo em termos de cumprimento de obrigações financeiras, como pagar contas de empréstimos, cartões de crédito, entre outros. Essa pontuação é calculada com base em diversas informações financeiras e de crédito do indivíduo, como histórico de pagamentos, níveis de endividamento, tempo de crédito, tipos de crédito utilizados, entre outros.

O objetivo de um modelo de credit score é prever o risco de um indivíduo se tornar inadimplente com suas obrigações financeiras. Em outras palavras, o modelo avalia a probabilidade de um indivíduo não cumprir com os pagamentos de empréstimos ou outros compromissos financeiros. Essa previsão é fundamental para instituições financeiras, como bancos e credores, na tomada de decisão sobre a concessão de crédito. Um modelo de credit score eficaz pode ajudar essas instituições a avaliar o risco de emprestar dinheiro a um determinado indivíduo e, assim, tomar decisões mais informadas sobre a aprovação ou negação de crédito, bem como sobre os termos e condições desses empréstimos.

**Atenção:** Notem que esse projeto é diferente da base que tenho trabalhado com vocês em aula, apesar de se tratar de uma base bancária durante a aula falamos sobre a variável Churn a ser prevista, nesse caso a previsão seria do valor do Score de Crédito.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px

In [ ]:
# Importação da base de dados
# Se o arquivo estiver em outra pasta, ajuste o caminho abaixo.
df = pd.read_csv("CREDIT_SCORE_PROJETO_PARTE1(1).csv", delimiter=';')

df.head(10)


,Age,Gender,Income,Education,Marital Status,Number of Children,Home Ownership,Credit Score
0,25.0,Female,"50.000,00",Bachelor's Degree,Single,0,Rented,High
1,30.0,Male,"100.000,00",Master's Degree,Married,2,Owned,High
2,35.0,Female,"75.000,00",Doctorate,Married,1,Owned,High
3,40.0,Male,"125.000,00",High School Diploma,Single,0,Owned,High
4,45.0,Female,"100.000,00",Bachelor's Degree,Married,3,Owned,High
5,50.0,Male,"150.000,00",Master's Degree,Married,0,Owned,High
6,26.0,Female,"40.000,00",Associate's Degree,Single,0,Rented,Average
7,31.0,Male,"60.000,00",Bachelor's Degree,Single,0,Rented,Average
8,NaN,Female,"80.000,00",Master's Degree,Married,2,Owned,High
9,NaN,Male,"105.000,00",Doctorate,Single,0,Owned,High


Legenda dos dados:

*   **Age** : Idade dos nossos clientes.

*   **Income** : Salário Mensal.

*   **Gender** : Gênero.

*   **Education** : Nível de escolaridade dos clientes.

*   **Marital** : Status Civilmente.

*   **Number of Children** : Quantidade de filhos.

*   **Home** : Tipo de residência, alugada ou própria.

*   **Credit Score** : Nossa variável preditora, o score de crédito dos clientes.


# Etapa 1: Relize os passos que vimos no módulo 14, de pré processamento dos dados.

**A) Verifique os tipos de dados, fazendo as transformações quando necessário.**


In [ ]:
# A) Verificando os tipos de dados
print(df.dtypes)

# Income está como texto porque os valores estão no formato brasileiro.
# Vamos transformar a coluna em número para permitir cálculos e análises.
df['Income'] = (
    df['Income']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

print('\nTipos após o tratamento:')
print(df.dtypes)


**B) Verifique se temos colunas com dados faltantes.
Caso existam colunas com dados faltantes faça o tratamento desses dados, excluindo ou substituindo esses valores. Justifique sua escolha.**

In [ ]:
# B) Verificando dados faltantes
print(df.isnull().sum())

# Age possui 34 valores ausentes. Como é uma variável numérica,
# vamos substituir os valores ausentes pela mediana da coluna.
# A mediana é adequada porque sofre menos influência de valores extremos.
mediana_idade = df['Age'].median()
df['Age'] = df['Age'].fillna(mediana_idade)

print('\nValores ausentes após o tratamento:')
print(df.isnull().sum())
print(f'\nMediana utilizada para Age: {mediana_idade}')


**C) Verifique se temos valores digitados de forma incorreta nas variáveis categóricas que necessitem de tratamento.**

In [ ]:
# C) Verificando possíveis inconsistências nas variáveis categóricas
colunas_categoricas = ['Gender', 'Education', 'Marital Status',
                       'Home Ownership', 'Credit Score']

for coluna in colunas_categoricas:
    print(f'\n{coluna}:')
    print(df[coluna].unique())

# Não foram identificadas grafias duplicadas ou categorias claramente incorretas.


# Etapa 2: Relize os passos que vimos no módulo 15, de análise.

**A) Realiza a análise univariada, aplique a função describe ao nosso dataframe para verificar os dados das variáveis numéricas, se encontrar a possível presença de outliers analise com gráficos a distribuição dos dados.Traga insights sobre os dados analisados.**

In [ ]:
# A) Análise univariada das variáveis numéricas
print(df.describe().round(2))

variaveis_numericas = ['Age', 'Income', 'Number of Children']

for coluna in variaveis_numericas:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x=coluna, bins=15, kde=True)
    plt.title(f'Distribuição de {coluna}')
    plt.xlabel(coluna)
    plt.ylabel('Quantidade de clientes')
    plt.tight_layout()
    plt.show()

# Verificação inicial de outliers pelo método do IQR
for coluna in variaveis_numericas:
    q1 = df[coluna].quantile(0.25)
    q3 = df[coluna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    outliers = df[(df[coluna] < limite_inferior) | (df[coluna] > limite_superior)]

    print(f'{coluna}: {len(outliers)} possível(is) outlier(s)')


**Insights:** A idade está concentrada principalmente entre 25 e 53 anos. O salário apresenta uma faixa ampla, de 25.000 a 162.500, sem outliers pelo IQR. A quantidade de filhos é predominantemente baixa, com a maioria dos clientes sem filhos.

**B) Agora realize a análise univariada para as variaveis categóricas, plote gráficos para entender a distribuição das categorias e tente retirar insights de cada gráfico.**

In [ ]:
# B) Análise univariada das variáveis categóricas
for coluna in colunas_categoricas:
    contagem = df[coluna].value_counts()

    plt.figure(figsize=(8, 4))
    ax = sns.barplot(x=contagem.index, y=contagem.values)
    plt.title(f'Distribuição de {coluna}')
    plt.xlabel(coluna)
    plt.ylabel('Quantidade de clientes')
    plt.xticks(rotation=20)

    for i, valor in enumerate(contagem.values):
        ax.text(i, valor + max(contagem.values) * 0.01, str(valor),
                ha='center', va='bottom')

    plt.tight_layout()
    plt.show()


**Insights:** A base possui 86 mulheres e 78 homens, portanto há uma distribuição relativamente equilibrada entre os gêneros. Entre as escolaridades, Bachelor's Degree é a categoria mais frequente. Quanto à moradia, a maioria dos clientes possui casa própria. O Credit Score é desbalanceado, com predominância de High.

**C) Você encontrou alguma coluna com outliers?
Se sim realize o tratamento desses casos.**

In [ ]:
# C) Identificação e tratamento dos possíveis outliers
# Pelo IQR, apenas Number of Children apresenta valores classificados
# estatisticamente como outliers: 5 clientes possuem 3 filhos.
#
# Porém, ter 3 filhos é um valor possível e não representa erro de cadastro.
# Por isso, não vamos excluir nem alterar esses registros.
# Age e Income não apresentam outliers pelo critério IQR.

def identificar_outliers_iqr(dataframe, coluna):
    q1 = dataframe[coluna].quantile(0.25)
    q3 = dataframe[coluna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    return dataframe[
        (dataframe[coluna] < limite_inferior) |
        (dataframe[coluna] > limite_superior)
    ]

for coluna in ['Age', 'Income', 'Number of Children']:
    outliers = identificar_outliers_iqr(df, coluna)
    print(f'{coluna}: {len(outliers)} outlier(s) pelo IQR')


**Insight sobre outliers:** O IQR sinaliza 5 registros com 3 filhos. Como esse valor é plausível e não indica erro de cadastro, os registros foram mantidos. O tratamento de outlier deve considerar o contexto do negócio, e não apenas o resultado matemático.

**D) Realize a análise Bivariada.
Tente responder as seguintes perguntas com gráficos seguidos de insights:**



*   Existe relação entre a idade e o status civil?
*   Qual a relação entre o score de crédito e o nível de escolaridade?
*  O salário parece influenciar na idade?
* O salário parece influenciar no Score de Crédito?
* Clientes com casa própria tendem a ter um score mais alto?



In [ ]:
# D) Análise bivariada

# 1. Existe relação entre a idade e o status civil?
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Marital Status', y='Age')
plt.title('Idade x Status Civil')
plt.xlabel('Status Civil')
plt.ylabel('Idade')
plt.show()

# 2. Qual a relação entre o score de crédito e o nível de escolaridade?
plt.figure(figsize=(9, 5))
sns.countplot(data=df, x='Education', hue='Credit Score')
plt.title('Escolaridade x Score de Crédito')
plt.xlabel('Escolaridade')
plt.ylabel('Quantidade de clientes')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# 3. O salário parece influenciar na idade?
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Age', y='Income')
plt.title('Idade x Salário')
plt.xlabel('Idade')
plt.ylabel('Salário')
plt.show()

print('Correlação entre idade e salário:',
      round(df['Age'].corr(df['Income']), 2))

# 4. O salário parece influenciar no Score de Crédito?
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Credit Score', y='Income')
plt.title('Salário x Score de Crédito')
plt.xlabel('Score de Crédito')
plt.ylabel('Salário')
plt.show()

# 5. Clientes com casa própria tendem a ter um score mais alto?
tabela_moradia_score = pd.crosstab(
    df['Home Ownership'],
    df['Credit Score'],
    normalize='index'
).mul(100).round(1)

print(tabela_moradia_score)

tabela_moradia_score.plot(kind='bar', figsize=(8, 5))
plt.title('Tipo de Moradia x Score de Crédito')
plt.xlabel('Tipo de Moradia')
plt.ylabel('Percentual (%)')
plt.xticks(rotation=0)
plt.legend(title='Score de Crédito')
plt.tight_layout()
plt.show()


**Insights da análise bivariada:** Clientes casados apresentam idade média maior que clientes solteiros. Há uma relação clara entre escolaridade e score: níveis como Master's e Doctorate concentram clientes com score High. Idade e salário apresentam correlação positiva de aproximadamente 0,69. O salário também varia bastante conforme o score, com valores mais altos entre clientes High. Na base analisada, clientes com casa própria apresentam proporção muito maior de score High do que clientes que moram de aluguel.

**E) Que outras perguntas te parecem fazer sentido explorarmos a resposta para conhecermos mais nossa base de dados e o comportamento dos clientes?**

 Elabore mais 3 perguntas e responda utilizando gráficos + insights.

In [ ]:
# E) Três perguntas adicionais

# Pergunta 1: Existe relação entre gênero e score de crédito?
tabela_genero_score = pd.crosstab(
    df['Gender'], df['Credit Score'], normalize='index'
).mul(100).round(1)

print('Gênero x Score de Crédito (%)')
print(tabela_genero_score)

tabela_genero_score.plot(kind='bar', figsize=(8, 5))
plt.title('Gênero x Score de Crédito')
plt.xlabel('Gênero')
plt.ylabel('Percentual (%)')
plt.xticks(rotation=0)
plt.legend(title='Score de Crédito')
plt.tight_layout()
plt.show()

# Pergunta 2: A quantidade de filhos está relacionada ao score?
tabela_filhos_score = pd.crosstab(
    df['Number of Children'], df['Credit Score'], normalize='index'
).mul(100).round(1)

print('Quantidade de filhos x Score de Crédito (%)')
print(tabela_filhos_score)

tabela_filhos_score.plot(kind='bar', figsize=(8, 5))
plt.title('Quantidade de Filhos x Score de Crédito')
plt.xlabel('Quantidade de filhos')
plt.ylabel('Percentual (%)')
plt.xticks(rotation=0)
plt.legend(title='Score de Crédito')
plt.tight_layout()
plt.show()

# Pergunta 3: O status civil está relacionado ao score?
tabela_estado_score = pd.crosstab(
    df['Marital Status'], df['Credit Score'], normalize='index'
).mul(100).round(1)

print('Status Civil x Score de Crédito (%)')
print(tabela_estado_score)

tabela_estado_score.plot(kind='bar', figsize=(8, 5))
plt.title('Status Civil x Score de Crédito')
plt.xlabel('Status Civil')
plt.ylabel('Percentual (%)')
plt.xticks(rotation=0)
plt.legend(title='Score de Crédito')
plt.tight_layout()
plt.show()


**Insights das perguntas adicionais:** Gênero apresenta diferenças menores no score quando comparado a outras variáveis. A quantidade de filhos mostra uma diferença importante entre clientes sem filhos e clientes com 1 ou mais filhos, embora as categorias de 3 filhos tenham poucos registros. O status civil apresenta relação mais evidente com o score: clientes casados concentram-se em High, enquanto clientes solteiros apresentam maior participação de Average e Low.

# Etapa 3: Relize os passos que vimos no módulo 17, de Correlação, Balanceamento, atributos categóricos e divisão base treino e teste.

**A) Vamos começar pela análise de correlação, plote da forma que achar melhor a análise de correlação, seja pela tabela ou pelo gráfico da matriz.**

In [ ]:
# A) Análise de correlação das variáveis numéricas
correlacao = df[['Age', 'Income', 'Number of Children']].corr().round(2)
print(correlacao)

plt.figure(figsize=(7, 5))
sns.heatmap(correlacao, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriz de Correlação - Variáveis Numéricas')
plt.tight_layout()
plt.show()


**Insight:** A principal relação linear entre as variáveis numéricas é Age x Income, com correlação positiva de aproximadamente 0,69. Number of Children apresenta correlações baixas com idade e salário.

**B) Você encontrou variáveis que tem uma média ou alta correlação? Se sim, quais? Te parece fazer sentido essas variáveis terem alta correlação? Justifique.**

In [ ]:
# B) Interpretação da correlação
# A maior correlação encontrada é entre Age e Income, aproximadamente 0,69.
# Isso indica uma relação positiva moderada/forte: clientes mais velhos
# tendem a apresentar salários maiores nesta base.
#
# A correlação entre Number of Children e as demais variáveis é baixa,
# indicando pouca relação linear com idade e salário.


**C) Temos muitos atributos categóricos nessa base, não? Vamos realizar a o tratamento desses atributos utilizando Label Encoder ou one hot. Após, exclua as colunas categóricas.**

In [ ]:
# C) Tratamento dos atributos categóricos
from sklearn.preprocessing import LabelEncoder

# Variáveis preditoras categóricas serão transformadas em One Hot Encoding.
# O Credit Score é a variável alvo e será codificado separadamente.
df_modelo = df.copy()

df_modelo = pd.get_dummies(
    df_modelo,
    columns=['Gender', 'Education', 'Marital Status', 'Home Ownership'],
    drop_first=True,
    dtype=int
)

label_encoder = LabelEncoder()
df_modelo['Credit Score'] = label_encoder.fit_transform(df_modelo['Credit Score'])

print('Classes do Credit Score:')
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

print('\nColunas após a codificação:')
print(df_modelo.columns.tolist())

df_modelo.head()


**D) Vamos plotar novamente a correlação, agora observando com as variáveis categóricas. Identifique se temos novas variáveis com forte correlação.**

In [ ]:
# D) Nova análise de correlação após o tratamento das variáveis categóricas
correlacao_modelo = df_modelo.corr().round(2)

plt.figure(figsize=(14, 10))
sns.heatmap(correlacao_modelo, annot=True, cmap='coolwarm', fmt='.2f',
            center=0, vmin=-1, vmax=1)
plt.title('Matriz de Correlação após Codificação')
plt.tight_layout()
plt.show()

# Correlações mais fortes com a variável alvo
correlacoes_score = (
    correlacao_modelo['Credit Score']
    .drop('Credit Score')
    .abs()
    .sort_values(ascending=False)
)

print('Variáveis com maior correlação absoluta com Credit Score:')
print(correlacoes_score.head(10))


**Insight:** Depois da codificação, aparecem relações relevantes entre algumas variáveis categóricas e o Credit Score. Isso é coerente com a análise bivariada: principalmente Home Ownership, Marital Status, Education e Income apresentam associação com o score. É importante lembrar que correlação não significa causalidade.

**F) Faça a separação da base em treino e teste e verifique utilizando shape:**

In [ ]:
# F) Separação da base em treino e teste
from sklearn.model_selection import train_test_split

X = df_modelo.drop(columns='Credit Score')
y = df_modelo['Credit Score']

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('X_treino:', X_treino.shape)
print('X_teste:', X_teste.shape)
print('y_treino:', y_treino.shape)
print('y_teste:', y_teste.shape)


**G) É hora de verificar se nossa coluna de Score de crédito está balanceada, verifique através de um gráfico e traga sua opinião acerca do balanceamento.**

In [ ]:
# G) Verificando o balanceamento do Credit Score
contagem_score = df['Credit Score'].value_counts()

plt.figure(figsize=(7, 4))
ax = sns.barplot(x=contagem_score.index, y=contagem_score.values)
plt.title('Distribuição do Credit Score')
plt.xlabel('Credit Score')
plt.ylabel('Quantidade de clientes')

for i, valor in enumerate(contagem_score.values):
    ax.text(i, valor + 2, str(valor), ha='center')

plt.tight_layout()
plt.show()

print((contagem_score / len(df) * 100).round(1))

# A base está desbalanceada: High representa a maior parte dos clientes,
# enquanto Low possui uma quantidade bem menor de observações.


**Insight:** O Credit Score está desbalanceado. A classe High possui 113 clientes, enquanto Average possui 36 e Low apenas 15. Como a classe Low é muito menor, um modelo pode ter dificuldade para aprender esse grupo sem uma técnica de balanceamento.

**H) Vamos realizar o balancecamento dos dados da coluna de credit score.**
Se lembre que realizazmos apenas para a base de treino.

In [ ]:
# H) Balanceamento da base de treino com SMOTE
from imblearn.over_sampling import SMOTE

print('Distribuição antes do SMOTE:')
print(y_treino.value_counts())

smote = SMOTE(random_state=42)
X_treino_balanceado, y_treino_balanceado = smote.fit_resample(
    X_treino, y_treino
)

print('\nDistribuição depois do SMOTE:')
print(y_treino_balanceado.value_counts())

print('\nShapes:')
print('X_treino_balanceado:', X_treino_balanceado.shape)
print('y_treino_balanceado:', y_treino_balanceado.shape)


**Insight:** O SMOTE foi aplicado somente à base de treino, preservando a base de teste para uma avaliação mais próxima de dados reais. Após o balanceamento, as três classes passam a ter a mesma quantidade de observações na base de treino.